# 1) Crop Router Pipeline (Colab)

Upload an image and run full VLM routing (SAM3 + BioCLIP) with visual detections.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datetime import datetime


def rt(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def is_repo_root(path: Path) -> bool:
    return (path / "src").is_dir() and (path / "config").is_dir() and (path / "scripts").is_dir()


def maybe_clone_repo() -> Path | None:
    if os.environ.get("AADS_DISABLE_AUTO_CLONE") == "1":
        return None

    repo_url = os.environ.get("AADS_REPO_URL", "https://github.com/EfeErim/bitirmeprojesi.git")
    clone_target = Path(os.environ.get("AADS_REPO_CLONE_TARGET", "/content/bitirmeprojesi")).expanduser()

    if is_repo_root(clone_target):
        return clone_target

    if clone_target.exists() and any(clone_target.iterdir()):
        for child in clone_target.iterdir():
            if child.is_dir() and is_repo_root(child):
                return child
        return None

    clone_target.parent.mkdir(parents=True, exist_ok=True)
    print(f"Repository not found locally. Auto-cloning from: {repo_url}")
    completed = subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(clone_target)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)

    if completed.returncode == 0 and is_repo_root(clone_target):
        return clone_target
    return None


def resolve_repo_root() -> Path:
    env_candidates = [os.environ.get("AADS_REPO_ROOT"), os.environ.get("REPO_ROOT")]
    for raw in env_candidates:
        if not raw:
            continue
        p = Path(raw).expanduser().resolve()
        if is_repo_root(p):
            return p

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if is_repo_root(p):
            return p

    common = [
        Path("/content/bitirme projesi"),
        Path("/content/bitirmeprojesi"),
        Path("/content/aads_ulora"),
        Path("/content/drive/MyDrive/bitirme projesi"),
        Path("/content/drive/MyDrive/bitirmeprojesi"),
    ]
    for p in common:
        if is_repo_root(p):
            return p

    auto_cloned = maybe_clone_repo()
    if auto_cloned is not None:
        return auto_cloned

    raise FileNotFoundError("Repository root not found and auto-clone failed. Set AADS_REPO_ROOT, or set AADS_REPO_URL/AADS_REPO_CLONE_TARGET.")


try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

rt("Cell 1: setup started")

if IN_COLAB:
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

req = ROOT / "colab_notebooks" / "requirements_colab.txt"
if req.exists() and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = "cpu"

rt(f"Repository root: {ROOT}")
rt(f"Device: {DEVICE}")
rt("Cell 1: setup completed")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from src.router.vlm_pipeline import VLMPipeline

rt("Cell 2: router initialization started")

config_path = ROOT / "config" / "base.json"
CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
rt("Config loaded")

ROUTER = VLMPipeline(CONFIG, device=DEVICE)
ROUTER.load_models()
rt("VLM models loaded")

profiles = list(CONFIG.get("router", {}).get("vlm", {}).get("profiles", {}).keys())
if "balanced" in profiles:
    default_profile = "balanced"
else:
    default_profile = profiles[0] if profiles else "balanced"

profile_dropdown = widgets.Dropdown(
    options=profiles if profiles else [default_profile],
    value=default_profile,
    description="Profile:",
)
apply_btn = widgets.Button(description="Apply Profile", button_style="info")
threshold_slider = widgets.FloatSlider(
    value=float(CONFIG.get("router", {}).get("vlm", {}).get("confidence_threshold", 0.25)),
    min=0.0,
    max=1.0,
    step=0.01,
    description="Conf:",
    continuous_update=False,
)
out = widgets.Output()
guide_html = widgets.HTML(value=(
    "<b>Control Guide (Suggested Values)</b><br>"
    "<ul style='margin-top:6px'>"
    "<li><b>Profile</b>: runtime behavior preset. Suggested default: <b>balanced</b>.</li>"
    "<li><b>Conf</b>: detection confidence threshold. Suggested start: <b>0.25</b>.</li>"
    "<li>Use <b>0.15-0.25</b> to catch more objects (higher recall, more false positives).</li>"
    "<li>Use <b>0.35-0.50</b> for stricter detections (higher precision, fewer boxes).</li>"
    "</ul>"
    "Click <i>Apply Profile</i> before running analysis cells."
))


def _apply_profile(_):
    with out:
        out.clear_output()
        applied = ROUTER.set_runtime_profile(profile_dropdown.value)
        print(f"Applied profile: {profile_dropdown.value} (applied={applied})")


apply_btn.on_click(_apply_profile)
_apply_profile(None)

display(guide_html)
display(widgets.HBox([profile_dropdown, apply_btn]))
display(threshold_slider)
display(out)
rt("Cell 2: router controls ready")

In [ ]:
from io import BytesIO

from PIL import Image

rt("Cell 3: image upload started")

UPLOADED_IMAGE = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded))
        UPLOADED_IMAGE = Image.open(BytesIO(uploaded[first_name])).convert("RGB")
except Exception as exc:
    print(f"Colab upload unavailable ({exc}). Using local fallback path if set.")
    fallback = os.environ.get("AADS_TEST_IMAGE")
    if fallback and Path(fallback).exists():
        UPLOADED_IMAGE = Image.open(fallback).convert("RGB")

if UPLOADED_IMAGE is None:
    raise RuntimeError("No image uploaded. Re-run this cell and upload an image.")

plt.figure(figsize=(7, 7))
plt.imshow(UPLOADED_IMAGE)
plt.axis("off")
plt.title("Uploaded Image")
plt.show()
rt("Cell 3: image upload completed")

In [ ]:
import pandas as pd
import matplotlib.patches as patches

rt("Cell 4: router analysis started")

if UPLOADED_IMAGE is None:
    raise RuntimeError("Upload an image first (Cell 3).")

try:
    analysis = ROUTER.analyze_image(
        UPLOADED_IMAGE,
        confidence_threshold=float(threshold_slider.value),
    )
except Exception as exc:
    raise RuntimeError(f"Router analysis failed: {exc}") from exc

rows = []
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(UPLOADED_IMAGE)
ax.axis("off")

for idx, det in enumerate(analysis.get("detections", []), start=1):
    crop = det.get("crop") or det.get("crop_label") or "unknown"
    part = det.get("part") or det.get("part_label") or "unknown"
    conf = float(det.get("crop_confidence", det.get("confidence", 0.0)))

    bbox = det.get("bbox") or det.get("box") or det.get("xyxy")
    if isinstance(bbox, dict):
        x = float(bbox.get("x", bbox.get("left", 0)))
        y = float(bbox.get("y", bbox.get("top", 0)))
        w = float(bbox.get("w", bbox.get("width", 0)))
        h = float(bbox.get("h", bbox.get("height", 0)))
        x1, y1, x2, y2 = x, y, x + w, y + h
    elif isinstance(bbox, (list, tuple)) and len(bbox) >= 4:
        x1, y1, x2, y2 = map(float, bbox[:4])
    else:
        x1 = y1 = x2 = y2 = 0.0

    rect = patches.Rectangle((x1, y1), max(1.0, x2 - x1), max(1.0, y2 - y1), linewidth=2, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, max(0, y1 - 6), f"{crop}/{part}: {conf:.2f}", color="lime", fontsize=9, bbox={"facecolor": "black", "alpha": 0.5, "pad": 1})

    rows.append({
        "Detection #": idx,
        "Crop": crop,
        "Part": part,
        "Confidence": round(conf, 4),
        "BBox": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)],
    })

plt.title("Router Detections")
plt.show()

if rows:
    display(pd.DataFrame(rows))
else:
    print("No detections returned by router.")

print("Processing time (ms):", analysis.get("processing_time_ms", 0.0))
rt(f"Cell 4: analysis completed with {len(rows)} detections")

In [ ]:
from src.pipeline.independent_multi_crop_pipeline import IndependentMultiCropPipeline

rt("Cell 5: full pipeline started")

if UPLOADED_IMAGE is None:
    raise RuntimeError("Upload an image first (Cell 3).")

pipeline = IndependentMultiCropPipeline(CONFIG, device=DEVICE)
pipeline.router = ROUTER
# Disable diagnostic analyzer to avoid duplicate router work in full-pipeline execution.
pipeline.router_analyzer = None

try:
    full_result = pipeline.process_image(UPLOADED_IMAGE, return_ood=True)
except Exception as exc:
    raise RuntimeError(f"Pipeline execution failed: {exc}") from exc
print("Pipeline result:")
print(json.dumps(full_result, indent=2, default=str))
rt("Cell 5: full pipeline completed")

In [ ]:
rt("Cell 6: rerun instructions displayed")
print("Run Again workflow:")
print("1) Re-run Cell 3 to upload a new image")
print("2) Re-run Cell 4 for router detections")
print("3) Optionally re-run Cell 5 for full pipeline output")